# 50장 · 인용 오류 수정과 회귀 검증

오류를 고친 뒤에는 원래 실패한 문항과 이전부터 성공하던 문항을 함께 실행합니다. 이를 통해 수정이 다른 답변에 미친 영향을 찾습니다.

p3을 검색하고도 p2를 인용한 후보를 수정합니다. 생성 단계에서 실제 검색 레코드의 ID를 사용하고, Q03과 기존 정상 질문을 다시 실행합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요.


## 후보에서 찾은 문단만 카드로 복구하기

49장에서 인용 오류로 좁힌 후보의 표시 규칙을 고칩니다. 수정한 카드가 같은 후보의 실제 검색 목록에 있으며 원문에도 남아 있는지 검사합니다.

이 프로젝트의 노트북은 한 폴더에서 열어 주세요. 첫 셀은 `team-project/midterm/chapter49.json`을 먼저 찾습니다. 팀 파일이 없으면 [이 장의 시작 예시](chapter50-start.json)를 노트북 옆에 내려받으세요. 출력 첫 줄에서 어느 파일을 읽었는지 확인할 수 있습니다.

In [ ]:
from pathlib import Path
import json
import re

here = Path.cwd()
project_root = here.parent if re.fullmatch(r"chapter[0-9]+", here.name) else here
team_file = project_root / "team-project/midterm/chapter49.json"
sample_file = here / "chapter50-start.json"
input_file = team_file if team_file.is_file() else sample_file
if not input_file.is_file():
    raise FileNotFoundError("이 장의 시작 예시를 내려받아 노트북 옆에 두세요.")
incoming = json.loads(input_file.read_text(encoding="utf-8"))
print("팀 기록" if input_file == team_file else "교재 예시", input_file.name)
print(incoming['citation_issues'])

교재 예시 chapter50-start.json
['Q03']


두 번째 줄은 읽은 파일의 `citation_issues` 값입니다.

## 검색한 원문에서 인용 생성

수정 기록 CH50-01｜입력은 Q03, 검색 정책은 error_keyword, 기대 출처는 course-guide-v1:p3입니다. 이전 생성기는 언제나 p2를 붙였습니다. 수정 후보는 원문 본문과 해당 문서 ID를 함께 읽습니다. 이 예시는 미리 작성한 생성 규칙을 고치는 사례입니다.


### 셀 01 · 검색 원문에서 답변 구성

가상 문서의 오류 대응 절차와 인용 ID를 같은 객체에서 꺼냅니다.


In [1]:
evidence = {'id': 'course-guide-v1:p3',
 'text': '제출 오류가 발생하면 오류 화면과 발생 시간을 기록하고 담당 강사에게 문의합니다. 비밀키가 보이는 화면은 공유하지 '
         '않습니다.',
 'source': '가상 과정 운영 안내 · 45일 수업용 페이지판 (source-pages.html)',
 'page': 1}
answer = f"모의 응답: {evidence['text']} [{evidence['id']}]"
print(answer)

모의 응답: 제출 오류가 발생하면 오류 화면과 발생 시간을 기록하고 담당 강사에게 문의합니다. 비밀키가 보이는 화면은 공유하지 않습니다. [course-guide-v1:p3]


화면·시간 기록, 강사 문의, 비밀키가 보이는 화면의 공유 금지까지 원문에 있습니다. 끝의 p3 인용이 올바른지와 문장의 뜻이 보존됐는지를 각각 확인하세요.


## 기존 성공 문항의 회귀 검사

새 답 한 건이 맞아도 기존 답변이 바뀌었을 수 있습니다. 제공 회귀표의 R50-Q01은 Q01의 수정 전후 결과를 기준선과 대조합니다. Q02의 점수 문의와 Q04의 주말 일정은 여전히 자료에서 확인할 수 없다고 안내해야 합니다.


### 셀 02 · 기존 정상 문항의 회귀 결과

기존 동작을 비교한 결과와 사람의 의미 검토 범위를 읽습니다.

Q01 답에 붙은 p2를 열고 ‘과제함’이라는 장소가 원문의 제출 위치와 같은지 읽으세요. ID가 맞는지는 코드로 검사할 수 있지만, 문장이 같은 뜻인지 판단한 사람만 meaning_review를 마쳤다고 적습니다.


In [2]:
regression = {'question_id': 'Q01',
 'same_A1_as_baseline': True,
 'observed_status_after': 'answered',
 'meaning_review': 'requires_direct_source_reading'}
print(regression['question_id'], regression['same_A1_as_baseline'], regression['observed_status_after'])

Q01 True answered


Q01의 응답 객체는 기준선과 같고 상태는 answered입니다. 팀에서 수정할 때는 문제 문항과 영향받지 않아야 할 문항을 함께 정하세요. 기준선에서 최종 후보까지는 검색과 생성 두 변경이 누적되므로 생성만 고친 비교는 보류 후보부터 시작합니다.

Q01이 다시 통과한 것은 수정 전부터 맞히던 문항을 보존했다는 증거입니다. 고친 오류가 해결됐다는 근거는 원래 실패한 문항에서 따로 얻어야 합니다. 두 결과를 분리해야 개선과 회귀를 한꺼번에 확인할 수 있습니다.


#### 인용 오류 하나를 고친 다음

<mark>수정 전에도 통과했던 문항을 같은 입력으로 다시 실행합니다.</mark> 실패 문항만 검사하면 출처 조립 방식의 변경이 다른 문서를 잘못 가리키는지 놓칩니다. 기존 성공 문항에서 답변 상태와 문단 ID가 함께 유지되는지 보세요.

선수 개념 · 46장 검색 규칙 변경과 실패 단계 진단

#### 수정한 실패 사례에서 기존 정상 문항으로 돌아가기

```mermaid
flowchart TD
nfix["오류 문항"]
  nbase["정상 문항"]
  nstatus["응답 상태"]
  nmeaning["원문 대조"]
  nfix --> nbase
  nbase --> nstatus
  nstatus --> nmeaning
```

오류 문항의 인용을 고친 뒤 기존에 성공하던 Q01을 다시 봅니다. 자동 비교 결과와 원문 의미 확인을 함께 읽으세요.

## 후보 원문에 없는 인용으로 회귀 실패 재현하기

50장의 생성 후보는 원문의 본문과 ID를 함께 사용합니다. 팀이 이 변경을 유지할 수 있도록, 원래 결함과 기존 성공 경로를 같은 검사 함수로 다시 실행합니다.

### 셀 03 · 근거와 인용이 어긋나면 실패하게 만들기

길이가 비슷한 두 ID가 읽기에는 자연스러워 보입니다. 생성 문안 안에 든 대괄호 인용을 꺼내 실제 검색 후보와 비교하세요.

In [3]:
import re

def backed_by_search(text, rows):
    cited = re.findall(r"\[([^\[\]]+)\]", text)
    return bool(cited) and set(cited).issubset({row["id"] for row in rows})

broken = "담당 강사에게 문의하세요. [course-guide-v1:p2]"
print(backed_by_search(broken, [evidence]))
print(backed_by_search(answer, [evidence]))

False
True


두 입력에 같은 검색 결과 p3을 넘겼습니다. 첫 문장의 인용만 p2이므로 False이고, 수정된 answer는 p3을 인용해 True입니다.

### 셀 04 · 종전에 성공하던 Q01과 무인용 답 검사

Q01은 p2가 정답입니다. 모든 답에 p3을 붙이는 수정이 통과하지 않도록 회귀 사례도 둡니다.

In [4]:
q01_doc = {"id": "course-guide-v1:p2"}
print(backed_by_search("과제함에 제출하세요. [course-guide-v1:p2]", [q01_doc]))
print(backed_by_search("자료에서 확인할 수 없습니다.", []))
mixed = "과제함에 제출하세요. [course-guide-v1:p2] [other-guide:p9]"
print(backed_by_search(mixed, [q01_doc]))

True
False
False


Q01은 그대로 통과합니다. 둘째 False는 근거 있는 답으로 채점할 수 없다는 결과입니다. 답변 불가가 기대되는 Q02라면 보류 상태를 별도 기준으로 채점합니다. 마지막 False는 p2를 맞게 인용했어도 후보에 없는 다른 문서를 덧붙여서는 안 된다는 뜻입니다.

<mark>인용 검사는 근거 없는 답을 유보하는 정책과 따로 둡니다.</mark> 보류 문장을 인용 검사에서 통과시키려고 임의의 문단을 붙이면 새로운 잘못된 근거가 생깁니다.

51장 인계서에는 Q03에서 실패했던 문자열과 수정 뒤 통과한 문자열을 함께 남기세요.

## 팀 사례에 적용하기 · 후보에서 찾은 문단만 카드로 복구하기

위 예제에 첫 셀에서 읽어 둔 팀 기록을 넣어 봅니다.

In [ ]:
fixed = []
for row in incoming["runs"]:
    ids = list(row["experiment_visible"])
    if row["id"] in incoming["citation_issues"]:
        ids = [key for key in row["experiment_ids"] if key.split(":")[-1] in incoming["documents"]]
    backed = set(ids).issubset(row["experiment_ids"])
    correct = [key.split(":")[-1] for key in ids] == row["expected"]
    fixed.append({**row, "fixed_ids": ids, "regression_ok": backed and correct})
new = {"runs": fixed, "documents": incoming["documents"], "fixes": incoming["citation_issues"],
       "decision": incoming["decision"], "evaluated_path": "실험 후보"}
print([(row["id"], row["regression_ok"]) for row in fixed])

[('Q01', True), ('Q02', True), ('Q03', True)]


Q03에 붙은 p3은 47장의 같은 후보 요청이 찾은 문단입니다. 검색하지 못한 행은 카드가 늘지 않습니다. 이 결과는 후보의 로컬 회귀이며 기준선 채택 판단과는 별개입니다.

In [ ]:
output_file = project_root / "team-project/midterm/chapter50.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open("a+", encoding="utf-8") as handle:
    handle.seek(0)
    previous_text = handle.read()
    if previous_text and json.loads(previous_text) != new:
        raise FileExistsError("같은 장의 다른 결과가 있습니다. 기존 JSON을 다른 이름으로 보관한 뒤 다시 실행하세요.")
    if not previous_text:
        json.dump(new, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
print(output_file.relative_to(project_root).as_posix())

team-project/midterm/chapter50.json


51장 첫 셀이 이 파일을 읽습니다. 입력을 바꿔 다시 실행할 때는 기존 결과 파일의 이름을 바꿔 보관하세요.

## 돌아보기

이 후보는 공개 질문 네 건의 로컬 회귀만 확인했습니다. 새로운 질문, 실제 모델, 동료 사용자에게도 같은 결과인지는 별도로 확인할 항목입니다.
